# 🧠 Transformers: Arquitectura, Matemática y Aplicaciones en NLP

En esta sesión exploraremos la arquitectura **Transformer**, su fundamento matemático, los bloques que la componen, y realizaremos un uso práctico con modelos preentrenados.


## 📜 De RNNs a Transformers

Hasta 2017, las arquitecturas dominantes para secuencias eran **RNN/LSTM/GRU**, con limitaciones: dificultad para dependencias largas, cálculo secuencial no paralelizable y problemas de gradiente.  
El artículo **["Attention is All You Need" (Vaswani et al., 2017)](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)** introduce el **Transformer**, que reemplaza la recurrencia por **mecanismos de atención** totalmente paralelizables.


## ⚙️ Arquitectura del Transformer

Bloques principales:
- **Encoder:** procesa la secuencia de entrada.
- **Decoder:** genera la secuencia de salida condicionada al encoder.

Subcapas dentro de cada bloque:
- **Self-Attention / Cross-Attention**
- **Feed-Forward (FFN)**
- **Residual + LayerNorm**

```
Input → [Embedding + Positional Encoding] → Encoder Layers → Context
Decoder Input → [Embedding + Positional Encoding] → Decoder Layers → Output
```

![Arquitectura de un transformer](https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer%2C_full_architecture.png)

# El encoder

## 🔢 Embeddings y Positional Encoding

Los Transformers reciben **vectores** y agregan información de orden mediante **Positional Encoding (PE)**:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

donde:
- $pos$ es la posición del token (0, 1, 2, …)
- $i$ es el índice de la dimensión del embedding
- $d_{model}$ es la dimensión total del embedding (p. ej., 512 en el paper)

En un Transformer, distinguimos claramente dos dimensiones:

**Longitud de secuencia ($n_{tokens}$)**:  
* cuántos elementos (palabras o sub-palabras) hay en la oración.
Ejemplo: "Los transformers son poderosos" → 4 tokens.

**Dimensión del embedding ($d_{model}$)**
* tamaño del vector numérico que representa cada token.
Ejemplo: $d_{model}$ = 512 (en el paper original) o 768 (en BERT-base).

En otras palabras, cada token se convierte en un vector de $d_{model}$ dimensiones.

Por tanto, la entrada al Transformer se representa como una matriz de tamaño:

$$X\in\mathbb{R}^{n_{tokens}\times d_{model}}$$

donde:
- cada fila corresponde a un token distinto,
- cada columna corresponde a una característica (dimensión del embedding).

📘 Ejemplo conceptual:
Si tienes 5 tokens y $d_{model}$ = 4, tu matriz de entrada se ve así:

| Token | $dim_0$ | $dim_1$ | $dim_2$ | $dim_3$ |
|:-----:|:----:|:----:|:----:|:----:|
| $T_1$    | 0.11 | 0.24 | 0.37 | 0.52 |
| $T_2$    | 0.85 | 0.41 | 0.09 | 0.76 |
| $T_3$    | 0.64 | 0.57 | 0.44 | 0.38 |
| $T_4$    | 0.19 | 0.82 | 0.60 | 0.71 |
| $T_5$    | 0.05 | 0.33 | 0.25 | 0.48 |

➡️ Cada token tiene 4 características, pero hay 5 tokens.

**¿Qué hace el Positional Encoding en este caso?**

Se genera una matriz del mismo tamaño que los embeddings:
$$PE\in\mathbb{R}^{n_{tokens}\times d_{modes}}$$
donde cada fila $PE_{pos}$ representa el vector de codificación posicional para la posición $pos$.

Luego se suma elemento a elemento:
$$X'=X+PE$$

In [1]:
#Ejemplo numérico pequeño

import numpy as np

n_tokens = 3
d_model = 4

# embeddings simulados
X = np.array([
    [0.2, 0.8, 0.4, 0.6],
    [0.5, 0.1, 0.3, 0.7],
    [0.9, 0.4, 0.2, 0.5]
])

# función de positional encoding
def positional_encoding(n_tokens, d_model):
    PE = np.zeros((n_tokens, d_model))
    for pos in range(n_tokens):
        for i in range(0, d_model, 2):
            angle = pos / np.power(10000, (2*i)/d_model)
            PE[pos, i] = np.sin(angle)
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(angle)
    return PE


In [2]:
PE = positional_encoding(n_tokens, d_model)
PE

array([[ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         1.00000000e+00],
       [ 8.41470985e-01,  5.40302306e-01,  9.99999998e-05,
         9.99999995e-01],
       [ 9.09297427e-01, -4.16146837e-01,  1.99999999e-04,
         9.99999980e-01]])

In [3]:
X_prime = X + PE
X_prime

array([[ 0.2       ,  1.8       ,  0.4       ,  1.6       ],
       [ 1.34147098,  0.64030231,  0.3001    ,  1.69999999],
       [ 1.80929743, -0.01614684,  0.2002    ,  1.49999998]])

## 🎯 El núcleo: Self-Attention
En una RNN, cada palabra "ve" solo lo que está antes de ella.
En un Transformer, cada palabra puede "atender" ("mirar") a cualquier otra palabra de la secuencia, con distinta intensidad.  
Esto se logra mediante el mecanismo de atención, que responde a la pregunta:  
>❓ ¿A qué otras palabras debo prestar más atención para entender el significado de la actual?

Por ejemplo, en la frase

> “El gato que estaba cansado se durmió.”

La palabra “se” debe atender fuertemente a “gato” (no a “cansado”) para entender la relación sujeto-verbo.

⚙️ **Definición matemática**

Para cada token se proyectan tres vectores:

- **Query ($Q$)**: lo que busca o pregunta este token.
- **Key ($K$)**: la “clave” de cada token (lo que ofrece).
- **Value ($V$)**: la información real que contiene.


A partir de estos tres, se calcula la atención:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

con:
- $QK^T$ mide la similitud entre cada token y los demás.
- $\sqrt{d_k}$ estabiliza los valores (evita softmax con valores extremos).
- El $softmax$ convierte esas similitudes en pesos de atención (que suman 1).
- Finalmente, multiplicamos por $V$ para obtener la mezcla ponderada de información.



In [4]:
W_Q = [[0.2, 0.1],
       [0.0, 0.3],
       [0.4, 0.1],
       [0.1, 0.2]]

W_K = [[0.3, 0.2],
       [0.1, 0.0],
       [0.0, 0.2],
       [0.4, 0.1]]

W_V = [[0.2, 0.4],
       [0.3, 0.1],
       [0.4, 0.2],
       [0.1, 0.3]]

In [5]:
Q = X_prime @ W_Q
Q

array([[0.36      , 0.92      ],
       [0.5583342 , 0.69624779],
       [0.59193948, 0.49610569]])

In [6]:
Q.shape[1]

2

In [7]:
d_k = Q.shape[1]

In [8]:
K = X_prime @ W_K
K

array([[0.88      , 0.28      ],
       [1.14647152, 0.4983142 ],
       [1.14117454, 0.55189948]])

In [9]:
V = X_prime @ W_V
V

array([[0.9       , 0.82      ],
       [0.75042489, 1.17063862],
       [0.58709543, 1.21214428]])

In [10]:
QKT = Q @ K.T
QKT

array([[0.5744    , 0.87117881, 0.91857036],
       [0.68628347, 0.98706441, 1.02141556],
       [0.65981634, 0.92585827, 0.94930674]])

In [11]:
S = QKT/np.sqrt(d_k)
S

array([[0.40616214, 0.61601644, 0.64952733],
       [0.4852757 , 0.69795994, 0.72224987],
       [0.46656061, 0.65468066, 0.67126123]])

In [12]:
def softmax(x, axis=-1):
    """
    Softmax estable numéricamente.
    Resta el máximo por 'axis' antes de exp para evitar overflow.
    """
    x = np.asarray(x, dtype=float)
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x_shift)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def attention_weights_from_scores(S, dk):
    S_scaled = S / np.sqrt(dk)       # escalado por sqrt(dk)
    A = softmax(S_scaled, axis=1)    # softmax fila a fila (cada fila suma 1)
    return A, S_scaled

In [13]:
A, S_scaled = attention_weights_from_scores(QKT, d_k)

print("S (QK^T) =\n", QKT)
print("\nS escalada (÷ sqrt(dk)) =\n", S_scaled)
print("\nPesos de atención (softmax por filas) =\n", A)
print("\nComprobación (sumas por fila):", A.sum(axis=1))


S (QK^T) =
 [[0.5744     0.87117881 0.91857036]
 [0.68628347 0.98706441 1.02141556]
 [0.65981634 0.92585827 0.94930674]]

S escalada (÷ sqrt(dk)) =
 [[0.40616214 0.61601644 0.64952733]
 [0.4852757  0.69795994 0.72224987]
 [0.46656061 0.65468066 0.67126123]]

Pesos de atención (softmax por filas) =
 [[0.28497882 0.3515209  0.36350028]
 [0.28535536 0.35298287 0.36166177]
 [0.29119406 0.35146494 0.35734101]]

Comprobación (sumas por fila): [1. 1. 1.]


In [14]:
A @ V

array([[0.73368032, 1.08580136],
       [0.73403693, 1.08559302],
       [0.73561596, 1.08336641]])

🧠 Recapitulando: una sola cabeza (“single-head”)

Lo que hicimos hasta ahora fue:

1. Tomar X' (tu embedding + positional encoding).
2. Calcular tres proyecciones lineales:
$$Q = X'W_Q,\ K = X'W_K,\ V=X'W_V$$
3. Obtener los pesos de atención:
 $$A = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

Combinar:

$$Z=AV$$

Eso se llama una cabeza de atención (attention head).
Es capaz de captar una sola relación o tipo de dependencia entre tokens (por ejemplo, relaciones sintácticas locales).


🔁 Extensión: Multi-Head Attention

El Transformer original no usa una sola cabeza, sino varias en paralelo, cada una con sus propios parámetros:

$$Q = X'W^Q_i,\ K = X'W^K_i,\ V=X'W^V_i\ i=1,2,\cdots,h$$

Cada cabeza produce su propia salida:
$$Z_i = \text{Attention}(Q_i,K_i,V_i)$$

Luego se concatenan todas las salidas:
$Z_{\text{concat}} = \text{Concat}(Z_1,\ Z_2,\ldots, Z_h)$
y se proyectan de nuevo al espacio de $d_{model}$:

$$\text{MultiHead}(X') = Z_{\text{concat}}W^O$$

💡 La idea...

Cada cabeza puede “mirar” diferentes patrones:
- Una puede enfocarse en dependencias cortas.
- Otra en relaciones largas.
- Otra en estructuras semánticas globales.  
Así, el modelo aprende distintas sub-espacialidades de atención simultáneamente.


🧮 Dimensiones típicas

Si $d_{model} = 512$ y $h = 8$ cabezas, entonces cada cabeza usa:

$$d_k=d_v=\frac{d_{model}}{h}=64$$

De esta forma, al concatenar las 8 salidas, recuperamos:

$$h\times d_v = 8 \times 64 = 512 = d_{model}$$

In [15]:
n_tokens, d_model = X_prime.shape
h = 2                    # número de cabezas
d_k = d_v = d_model // h # 2 dimensiones por cabeza

In [16]:
np.random.seed(42)  # para reproducibilidad

# Cada cabeza tiene su propio W_Q, W_K, W_V de tamaño (d_model x d_k)
W_Q = [np.random.uniform(-0.5, 0.5, (d_model, d_k)) for _ in range(h)]
W_K = [np.random.uniform(-0.5, 0.5, (d_model, d_k)) for _ in range(h)]
W_V = [np.random.uniform(-0.5, 0.5, (d_model, d_v)) for _ in range(h)]

# Matriz de salida (concatena todas las cabezas)
W_O = np.random.uniform(-0.5, 0.5, (h * d_v, d_model))

In [17]:
def scaled_dot_attention(Q, K, V):
    """Cálculo de self-attention por una cabeza."""
    dk = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(dk)
    # Softmax estable numéricamente
    e_x = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn = e_x / np.sum(e_x, axis=-1, keepdims=True)
    Z = attn @ V
    return Z, attn

In [18]:
def multi_head_attention(X, W_Q, W_K, W_V, W_O):
    """Ejecuta h cabezas de atención en paralelo y las concatena."""
    heads_output = []
    attn_maps = []

    for Wq, Wk, Wv in zip(W_Q, W_K, W_V):
        Q = X @ Wq
        K = X @ Wk
        V = X @ Wv
        Z, attn = scaled_dot_attention(Q, K, V)
        heads_output.append(Z)
        attn_maps.append(attn)

    # concatenar las salidas de todas las cabezas
    Z_concat = np.concatenate(heads_output, axis=-1)

    # proyección final
    output = Z_concat @ W_O
    return output, attn_maps

In [19]:
Z_multi, attn_weights = multi_head_attention(X_prime, W_Q, W_K, W_V, W_O)

print("Salida final del bloque Multi-Head (Z_multi):")
print(Z_multi.round(4))

print("\nMapas de atención (una por cabeza):")
for i, attn in enumerate(attn_weights):
    print(f"\nCabeza {i+1}:")
    print(attn.round(3))

Salida final del bloque Multi-Head (Z_multi):
[[0.6812 0.4605 0.7607 0.387 ]
 [0.6823 0.4824 0.7313 0.3766]
 [0.6822 0.4881 0.7208 0.3751]]

Mapas de atención (una por cabeza):

Cabeza 1:
[[0.278 0.341 0.381]
 [0.241 0.344 0.415]
 [0.233 0.344 0.422]]

Cabeza 2:
[[0.388 0.318 0.294]
 [0.373 0.324 0.302]
 [0.358 0.329 0.313]]


## 🧱 Subcapa Feed-Forward y Conexiones Residuales

Esta etapa asegura estabilidad, normalización y permite que el modelo aprenda transformaciones más complejas sobre las representaciones contextualizadas que produjo la atención.

🧱 1️⃣ Residual Connection (conexión residual)

La idea viene de las ResNets:
$$\text{Output}=X+\text{Sublayer}(X)$$

En este caso:
$X$ es la entrada original del bloque (`X_prime`), $Sublayer(X)$ es la salida del bloque de Multi-Head Attention ($Z_{multi}$).

Esto permite que la red conserve la información original, evitando la degradación del gradiente en redes profundas.

⚖️ 2️⃣ Layer Normalization

Después de la suma residual, se aplica una normalización por capa (LayerNorm):

$$LayerNorm(x_i) = \frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}\cdot\gamma+\beta}$$

donde:
- $\mu$ y $\sigma^2$ son la media y varianza por vector (por token),
- $\gamma$ y $\beta$ son parámetros aprendibles que re-escalan y desplazan.

Esto estabiliza el rango de activaciones entre capas y acelera el entrenamiento.

🧮 3️⃣ Feed-Forward Network (FFN)

Cada token pasa por una red totalmente conectada, independiente del resto:
$$FNN(x) = \max(0,xW_1+b1)W_2+b_2$$
donde:
- $W_1$ expande la dimensionalidad (típicamente 4$\times d_{model}$),
- $W_2$ la reduce de nuevo a $d_{model}$.

Por ejemplo, si $d_{model}$ = 4, usamos internamente una capa oculta de $d_{ff} = 8$ (solo para ilustración).

In [20]:
def layer_norm(X, eps=1e-6):
    """Normalización por capa (LayerNorm)."""
    mean = X.mean(axis=-1, keepdims=True)
    var = ((X - mean) ** 2).mean(axis=-1, keepdims=True)
    norm = (X - mean) / np.sqrt(var + eps)
    return norm  # gamma y beta se omiten para simplicidad

def feed_forward(X, d_ff=8):
    """Pequeña red feed-forward aplicada por token."""
    n_tokens, d_model = X.shape
    # pesos de ejemplo reproducibles
    np.random.seed(10)
    W1 = np.random.uniform(-0.5, 0.5, (d_model, d_ff))
    b1 = np.random.uniform(-0.1, 0.1, (d_ff,))
    W2 = np.random.uniform(-0.5, 0.5, (d_ff, d_model))
    b2 = np.random.uniform(-0.1, 0.1, (d_model,))
    # activación ReLU
    hidden = np.maximum(0, X @ W1 + b1)
    out = hidden @ W2 + b2
    return out


In [21]:
# --- 1️⃣ Residual connection ---
residual_1 = X_prime + Z_multi   # X' + MultiHead(X')

# --- 2️⃣ Layer normalization ---
norm_1 = layer_norm(residual_1)

# --- 3️⃣ Feed Forward ---
ff_output = feed_forward(norm_1)

# --- 4️⃣ Segundo residual + layer norm ---
residual_2 = norm_1 + ff_output
norm_2 = layer_norm(residual_2)

print("Salida final del bloque Encoder (norm_2):")
print(norm_2.round(4))

Salida final del bloque Encoder (norm_2):
[[-1.1464  1.2909 -0.7825  0.638 ]
 [ 1.0771 -0.7843 -1.1909  0.8981]
 [ 1.4349 -1.0346 -0.8289  0.4286]]


🧭 5️⃣ Qué ocurre en el flujo del diagrama

1. Multi-Head Attention contextualiza los tokens.
2. Residual 1 + LayerNorm estabiliza y conserva información original.
3. Feed-Forward aplica transformaciones no lineales por token.
4. Residual 2 + LayerNorm produce la salida final del encoder layer.
5. Cada Encoder Layer tiene esta misma estructura, y se apilan (6 en el Transformer original) para lograr mayor profundidad contextual.

# El decoder
🧩 1️⃣ Estructura general del Decoder

Cada capa del Decoder tiene tres sub-bloques principales:

1. Masked Self-Attention
→ El modelo se “autoatiende” como en el encoder, pero sin mirar al futuro.

2. Cross-Attention (Encoder–Decoder Attention)
→ Atiende a la salida del encoder (contexto de entrada).

3. Feed-Forward Network + Residual + LayerNorm
→ Igual al encoder.

```
y_prev → [Masked Self-Attention]
         ↓
     + [Cross-Attention con Encoder Output]
         ↓
     + [Feed Forward + Norm + Residual]
         ↓
     Predicción siguiente token

```


🚫 2️⃣ Masked Self-Attention

Durante la generación de texto, el modelo no debe ver tokens futuros.
Por ejemplo, si estamos prediciendo el token 3, no puede usar información del token 4.

Esto se logra aplicando una máscara triangular sobre la matriz de similitudes $QK^T$, antes del softmax:

$$ scores_{ij}=\left\{ \begin{array}{lcc} \frac{(QK^T)_{ij}}{\sqrt{d_k}} & si & j\leq i \\  \\ -\infty & si & j > i \end{array}
\right. $$


Así, el softmax ignora los elementos futuros (porque $e^{-\infty}=0$).

🧮 3️⃣ Cross-Attention (Encoder–Decoder Attention)

Después del masked self-attention, el Decoder usa otra atención,
pero esta vez:

- las queries ($Q$) vienen del Decoder,
- las keys ($K$) y values ($V$) provienen del **Encoder Output**.

Esto permite que cada token generado mire hacia el contexto codificado de la entrada.
$$\text{CrossAttention}(Q_{dec},K_{enc},V_{enc})=\text{softmax}\left(\frac{Q_{dec}
 K^T_{enc}}{\sqrt{d_k}}\right)$$

In [22]:
# ============================================
# 1️⃣ Salida del encoder (contexto)
# ============================================
encoder_output = norm_2.copy()   # salida final del encoder
n_tokens_enc, d_model = encoder_output.shape

In [23]:
# ============================================
# 2️⃣ Entrada del decoder (supongamos 3 tokens previos generados)
# ============================================
Y_in = np.array([
    [0.1, 0.3, 0.2, 0.4],  # <s> (inicio)
    [0.2, 0.5, 0.3, 0.6],  # palabra 1
    [0.3, 0.7, 0.4, 0.8]   # palabra 2
])
n_tokens_dec = Y_in.shape[0]
h = 2
d_k = d_v = d_model // h

In [24]:
# ============================================
# 3️⃣ Máscara triangular inferior (para masked self-attention)
# ============================================
def causal_mask(n):
    mask = np.triu(np.ones((n, n)), k=1)  # superior a diagonal
    mask = np.where(mask == 1, -np.inf, 0.0)
    return mask

mask = causal_mask(n_tokens_dec)
print("Máscara causal:\n", mask)

Máscara causal:
 [[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]]


In [25]:
# ============================================
# 4️⃣ Función de atención con máscara opcional
# ============================================
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_attention(Q, K, V, mask=None):
    dk = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(dk)
    if mask is not None:
        scores = scores + mask  # aplica la máscara
    A = softmax(scores, axis=1)
    Z = A @ V
    return Z, A

# ============================================
# 5️⃣ Masked Self-Attention del decoder
# ============================================
np.random.seed(123)
W_Q_dec = [np.random.uniform(-0.5, 0.5, (d_model, d_k)) for _ in range(h)]
W_K_dec = [np.random.uniform(-0.5, 0.5, (d_model, d_k)) for _ in range(h)]
W_V_dec = [np.random.uniform(-0.5, 0.5, (d_model, d_v)) for _ in range(h)]
W_O_dec = np.random.uniform(-0.5, 0.5, (h*d_v, d_model))

In [26]:
def multi_head_attention_cross(X_q, X_kv, W_Q, W_K, W_V, W_O, mask=None):
    """
    X_q: (n_dec, d_model)  -> queries desde el decoder
    X_kv: (n_enc, d_model) -> keys/values desde el encoder
    W_Q, W_K, W_V: listas de longitud h con matrices (d_model, d_k) o (d_model, d_v)
    W_O: (h*d_v, d_model)
    """
    heads_out, attn_maps = [], []
    for Wq, Wk, Wv in zip(W_Q, W_K, W_V):
        Q = X_q @ Wq              # (n_dec, d_k)
        K = X_kv @ Wk             # (n_enc, d_k)
        V = X_kv @ Wv             # (n_enc, d_v)
        Z, A = scaled_dot_attention(Q, K, V, mask=mask)  # atención encoder–decoder
        heads_out.append(Z)       # (n_dec, d_v)
        attn_maps.append(A)       # (n_dec, n_enc)
    Z_concat = np.concatenate(heads_out, axis=-1)        # (n_dec, h*d_v)
    out = Z_concat @ W_O                                  # (n_dec, d_model)
    return out, attn_maps

# 1er sub-bloque: masked self-attention
masked_output, masked_attn = multi_head_attention(Y_in, W_Q_dec, W_K_dec, W_V_dec, W_O_dec)

# ============================================
# 6️⃣ Cross-Attention con salida del encoder
# ============================================
np.random.seed(321)
W_Q_cross = [np.random.uniform(-0.5, 0.5, (d_model, d_k)) for _ in range(h)]
W_K_cross = [np.random.uniform(-0.5, 0.5, (d_model, d_k)) for _ in range(h)]
W_V_cross = [np.random.uniform(-0.5, 0.5, (d_model, d_v)) for _ in range(h)]
W_O_cross = np.random.uniform(-0.5, 0.5, (h*d_v, d_model))

In [27]:
cross_output, cross_attn = multi_head_attention_cross(
    X_q=masked_output,         # Q desde el decoder
    X_kv=encoder_output,       # K,V desde el encoder
    W_Q=W_Q_cross,
    W_K=W_K_cross,
    W_V=W_V_cross,
    W_O=W_O_cross,
    mask=1                  # no se enmascara en cross-attention
)
print("Masked Self-Attention Output:\n", masked_output.round(4))
print("\nCross-Attention Output:\n", cross_output.round(4))

Masked Self-Attention Output:
 [[ 0.0199  0.0373 -0.0527  0.0758]
 [ 0.0199  0.0373 -0.0527  0.0758]
 [ 0.0199  0.0373 -0.0527  0.0759]]

Cross-Attention Output:
 [[-0.0577  0.1115 -0.0933 -0.2179]
 [-0.0577  0.1115 -0.0933 -0.2179]
 [-0.0577  0.1115 -0.0933 -0.2179]]


1. Máscara causal (mask)
Evita que el token en la posición i use información de tokens posteriores (j > i).

2. Masked Self-Attention
Cada token del decoder mira solo hacia atrás en la secuencia generada.

3. Cross-Attention
El decoder ahora usa encoder_output como contexto externo:

- $Q$ viene del decoder,
- $K$, $V$ vienen del encoder.

4. Feed-Forward + Residual + LayerNorm
(Exactamente igual al encoder; se aplica después de las atenciones.)

# Sobre la salida

🧠 1️⃣ **Concepto general**: del decoder al vocabulario

La salida del decoder (por ejemplo, `cross_output`) tiene forma:
$$Y_{dec}\in\mathbb{R}^{n_{tokens}\times d_{model}}$$

Para convertirla en probabilidades sobre las palabras del vocabulario, aplicamos una `capa lineal` + `softmax`:

$$P(token) = \text{softmax}(Y_{dec}W^T_{out}+b)$$

donde:

- $W_{\text{out}} \in \mathbb{R}^{|V| \times d_{\text{model}}}$
proyecta cada vector del decoder al tamaño del vocabulario.
- $|V|$ es el tamaño del vocabulario (p. ej. 10 000 o 50 000 tokens).
- $b$ es un sesgo opcional.

El resultado es una distribución de probabilidad sobre todas las palabras posibles del vocabulario, para cada posición del decoder.


🔢 2️⃣ **Cálculo paso a paso**

Supongamos que: el tamaño del vocabulario es pequeño ($|V| = 6$) solo para visualizar, y `cross_output` es la última salida del decoder (de dimensión 4).

La operación es:

$$\text{logits} = Y_{dec}W^T_{out}+b$$

Luego

$$P=\text{softmax(logits)}$$

El token con mayor probabilidad (máximo en $P$) es el siguiente que el modelo generará.

In [28]:
# Salida del decoder (por ejemplo, cross_output de la celda anterior)
Y_dec = cross_output

# Definimos un vocabulario pequeño de 6 palabras
vocab = ["hola", "mundo", "transformer", "gato", "naranja", "fin"]
V = len(vocab)
d_model = Y_dec.shape[1]

# Pesos de proyección hacia el vocabulario
np.random.seed(555)
W_out = np.random.uniform(-0.5, 0.5, (V, d_model))
b_out = np.random.uniform(-0.1, 0.1, (V,))

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)  # estabilidad numérica
    e_x = np.exp(x)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

In [29]:
# Capa lineal
logits = Y_dec @ W_out.T + b_out

# Distribuciones de probabilidad
probs = softmax(logits, axis=1)

print("Logits (sin softmax):")
print(logits.round(7))

print("\nProbabilidades (softmax):")
print(probs.round(7))


Logits (sin softmax):
[[-0.1657639 -0.1472502  0.0997979  0.0132395  0.0176635  0.0175418]
 [-0.1657636 -0.1472496  0.0997979  0.0132397  0.0176642  0.0175416]
 [-0.1657633 -0.147249   0.0997979  0.0132399  0.0176649  0.0175414]]

Probabilidades (softmax):
[[0.1444798 0.1471796 0.1884252 0.1728014 0.1735676 0.1735464]
 [0.1444798 0.1471796 0.1884252 0.1728014 0.1735676 0.1735464]
 [0.1444798 0.1471797 0.1884252 0.1728014 0.1735677 0.1735463]]


In [30]:
# Token predicho (por posición)
pred_indices = np.argmax(probs, axis=1)
pred_tokens = [vocab[i] for i in pred_indices]

print("\nTokens predichos por el decoder:")
for t, p in zip(pred_tokens, probs[np.arange(len(probs)), pred_indices]):
    print(f"{t:12s} (p = {p:.3f})")


Tokens predichos por el decoder:
transformer  (p = 0.188)
transformer  (p = 0.188)
transformer  (p = 0.188)


## 📊 Complejidad y ventajas

- RNNs: cómputo secuencial $O(n)$ sin paralelismo.
- Transformers: atención con costo $O(n^2)$ **pero totalmente paralelizable** en GPUs/TPUs.

**Ventajas:** capturan dependencias largas, permiten *pretraining* a gran escala y *transfer learning* eficiente.


## 🧬 Modelos derivados y extensiones del Transformer

Desde **Vaswani et al., 2017**, surgieron variantes enfocadas en comprensión, generación o tareas secuencia-a-secuencia: **BERT**, **GPT**, **BART**, **T5**, **DistilBERT**, **ALBERT**, entre otras.


In [40]:
!pip install -U "transformers<5" accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from transformers import pipeline, set_seed
import torch, textwrap

# =========================================================
# Configuración general
# =========================================================
set_seed(42)
device = 0 if torch.cuda.is_available() else -1

print(f"Device: {'CUDA' if device==0 else 'CPU'}")


Device: CPU


## 🧩 BERT — Bidirectional Encoder Representations from Transformers

**Empresa / patrocinio:** Google AI (2018)

### 🧠 Cómo se entrenó
BERT fue entrenado usando un **enfoque auto-supervisado** a gran escala sobre un corpus de texto enorme (Wikipedia + BookCorpus), con dos tareas principales:
1. **Masked Language Modeling (MLM):** se ocultan aleatoriamente el 15% de las palabras, y el modelo debe predecirlas a partir del contexto a ambos lados.
2. **Next Sentence Prediction (NSP):** se le dan pares de oraciones y debe predecir si la segunda sigue a la primera.

### ⚙️ Enfoque arquitectónico
- Basado **solo en el encoder** del Transformer.
- Entrenamiento **bidireccional**, es decir, cada token ve contexto de izquierda y derecha.
- Dimensiones típicas: BERT-base (12 capas, 768D), BERT-large (24 capas, 1024D).

### 💪 Fortalezas
- Captura relaciones contextuales profundas (semánticas y sintácticas).
- Excelente rendimiento en tareas de *understanding*: clasificación, NER, QA, análisis de sentimientos.
- Punto de partida para múltiples variantes: RoBERTa, ALBERT, DistilBERT, etc.

### ⚠️ Debilidades
- No puede **generar texto** (solo comprenderlo).
- Costoso de entrenar y ajustar (requiere GPUs extensas).
- NSP se demostró poco útil; fue eliminado en versiones posteriores como RoBERTa.


| Aspecto| Detalle|
| :----------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Idioma original**| Inglés|
| **Corpus de entrenamiento**| Wikipedia + BookCorpus (solo inglés)|
| **Desempeño en español**| Bajo si usas `bert-base-uncased` directamente; el vocabulario (WordPiece) no fue entrenado con tokens españoles.|
| **Alternativas recomendadas para español** | 🟢 **BETO** (`dccuchile/bert-base-spanish-wwm-cased`): entrenado 100% en español sobre Wikipedia + OPUS.  <br>🟢 **mBERT (Multilingual BERT)**: modelo multilingüe entrenado en 104 idiomas. |
| **Conclusión**| Usa **BETO** si tu tarea es exclusivamente en español; **mBERT** si es multilingüe.|


In [2]:
fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased",
    device=device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if 

In [3]:
sent_mlm = "Transformers are [MASK] models for natural language processing."
mlm_out = fill_mask(sent_mlm, top_k=5)

In [4]:
print("\n" + "="*70)
print("1) BERT (Masked LM) — completa la palabra en contexto")
print("Prompt:", sent_mlm)


1) BERT (Masked LM) — completa la palabra en contexto
Prompt: Transformers are [MASK] models for natural language processing.


In [5]:
for i, cand in enumerate(mlm_out, 1):
    print(f"{i:>2}. {cand['sequence']}  (p={cand['score']:.4f})")

 1. transformers are software models for natural language processing.  (p=0.1382)
 2. transformers are computational models for natural language processing.  (p=0.0820)
 3. transformers are mathematical models for natural language processing.  (p=0.0669)
 4. transformers are computer models for natural language processing.  (p=0.0656)
 5. transformers are theoretical models for natural language processing.  (p=0.0125)


## 💬 GPT — Generative Pre-trained Transformer

**Empresa / patrocinio:** OpenAI (2018)

### 🧠 Cómo se entrenó
GPT se entrena de forma **autoregresiva**: el modelo predice el siguiente token a partir de los anteriores.  
Su función objetivo es maximizar:
$$
P(w_1, w_2, ..., w_n) = \prod_{t=1}^{n} P(w_t \mid w_{<t})
$$

El modelo se preentrena en grandes corpus no etiquetados (Internet, libros, artículos, código) y luego se ajusta mediante *fine-tuning* o *RLHF* (Refuerzo con Retroalimentación Humana) para comportamientos conversacionales.

### ⚙️ Enfoque arquitectónico
- Basado **solo en el decoder** del Transformer.
- Usa **máscara causal** para no mirar tokens futuros.
- Escalable: GPT-1 (117M parámetros), GPT-2 (1.5B), GPT-3 (175B), GPT-4 y GPT-5 (modelos multimodales).

### 💪 Fortalezas
- Capaz de generar texto fluido, coherente y contextual.
- Excelente en tareas de **creación, resumen, diálogo y programación**.
- Aprendizaje en contexto (*in-context learning*).

### ⚠️ Debilidades
- No “entiende” bidireccionalmente el texto (solo pasado → futuro).
- Puede generar información incorrecta (*alucinaciones*).
- Costos computacionales y energéticos altísimos en entrenamiento.

| Aspecto | Detalle|
| :----------------------------------------- | :----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Idioma original**| Inglés|
| **Corpus de entrenamiento**| Textos de Internet mayoritariamente en inglés (WebText, Common Crawl, libros, código)|
| **Desempeño en español**| GPT-2 y GPT-3 comprenden español razonablemente, pero con sesgos y errores gramaticales; la fluidez es buena, la coherencia semántica menor.|
| **Alternativas recomendadas para español** | 🟢 **GPT-NeoX / GPT-J multilingües** (EleutherAI).<br>🟢 **GPT-2 Spanish** (`datificate/gpt2-small-spanish` o `DeepESP/gpt2-spanish`) entrenados con corpus español.|
| **Conclusión**| GPT-4/GPT-5 (los modelos comerciales de OpenAI) sí son **multilingües potentes**, pero los GPT abiertos (p. ej. GPT-2) funcionan mejor en inglés a menos que uses una versión *fine-tuned* en español. |


In [6]:
gen = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

Device set to use cpu


In [7]:
prompt = "The Transformer architecture changed NLP because"
gen_out = gen(prompt, max_length=60, do_sample=True, temperature=0.9, top_p=0.9, num_return_sequences=1)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [8]:
print("\n" + "="*70)
print("2) GPT-2 (Generación de texto) — continuación creativa")
print("Prompt:", prompt)
print(textwrap.fill(gen_out[0]["generated_text"], width=100))


2) GPT-2 (Generación de texto) — continuación creativa
Prompt: The Transformer architecture changed NLP because
The Transformer architecture changed NLP because of the high frequency bandgap in the spectrum,
which is why the low frequency bandgap was reduced. The frequency of the low frequency bandgap was
reduced because of the lower bandgap which is why the high frequency bandgap was increased. It is
important to note that the lower frequency bandgap (0-3 ms) was due to the lower frequency bandgap
(0-4 ms) which was due to the higher frequency bandgap (5-12 ms). These differences have been
accounted for in a number of studies (5, 6, 7, 8, 9–12, 13–18, 19–24). It has been shown that the
frequency difference between a spectrum of 1 MHz and 5 MHz is about 30% (22, 25) which is because
the lower frequency bandgap is reduced and this decreases the frequency difference (21–29) (24, 26).
2.4.2. Frequency bands  The frequency band is the range between frequencies which represent the full
spe

## 🔄 BART — Bidirectional and Auto-Regressive Transformers

**Empresa / patrocinio:** Meta AI (Facebook AI, 2019)

### 🧠 Cómo se entrenó
BART se entrenó como un **autoencoder denoising**: se corrompe un texto de entrada (eliminando, barajando o enmascarando fragmentos) y el modelo debe **reconstruir el texto original**.

Esto combina las ventajas de:
- Un encoder bidireccional (como BERT),
- y un decoder autoregresivo (como GPT).

### ⚙️ Enfoque arquitectónico
- Arquitectura **Encoder–Decoder completa**.
- Se usa en tareas *seq2seq* como resumen, traducción, corrección gramatical o generación controlada.

### 💪 Fortalezas
- Excelente en *text summarization* y *paraphrasing*.
- Generaliza bien en múltiples tareas supervisadas.
- Compatible con *fine-tuning multitarea*.

### ⚠️ Debilidades
- Modelo grande, con tiempos de inferencia altos.
- Menor rendimiento en comprensión pura que BERT.
- Dificultad para inferencia en dispositivos limitados.

| Aspecto| Detalle|
| :----------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **Idioma original**| Inglés|
| **Corpus de entrenamiento**| News + Wikipedia en inglés (CNN/DailyMail)|
| **Desempeño en español**| Limitado si usas `facebook/bart-large-cnn`; entiende poco el español porque su tokenizer fue ajustado al inglés.|
| **Alternativas recomendadas para español** | 🟢 **mBART-50** (`facebook/mbart-large-50`) o **mBART-25**: multilingües (incluyen español, francés, portugués, etc.).<br>Estos modelos pueden hacer traducción, resumen y generación en español. |
| **Conclusión**| Usa **mBART-50** para cualquier tarea *seq2seq* en español.|


In [9]:
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=device
)

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [10]:
doc = (
    "Transformers have revolutionized natural language processing by replacing recurrence with "
    "self-attention, enabling massive parallelization. This architecture allows models to capture "
    "long-range dependencies more effectively, which is crucial for tasks like translation, "
    "summarization, and question answering. By pretraining on large corpora and fine-tuning on specific "
    "tasks, Transformer-based models achieve state-of-the-art results across a wide range of benchmarks."
)

sum_out = summarizer(doc, max_length=60, min_length=20, do_sample=False)

In [11]:
print("\n" + "="*70)
print("3) BART / DistilBART (Resumen) — condensación de texto")
print("Texto original:")
print(textwrap.fill(doc, width=100))


3) BART / DistilBART (Resumen) — condensación de texto
Texto original:
Transformers have revolutionized natural language processing by replacing recurrence with self-
attention, enabling massive parallelization. This architecture allows models to capture long-range
dependencies more effectively, which is crucial for tasks like translation, summarization, and
question answering. By pretraining on large corpora and fine-tuning on specific tasks, Transformer-
based models achieve state-of-the-art results across a wide range of benchmarks.


In [12]:
print("\nResumen:")
print(textwrap.fill(sum_out[0]["summary_text"], width=100))


Resumen:
 Transformer-based models achieve state-of-the-art results across a wide range of benchmarks . This
architecture allows models to capture long-range dependencies more effectively, which is crucial for
tasks like translation, summarization, and question answering .


## 🧮 T5 — Text-to-Text Transfer Transformer

**Empresa / patrocinio:** Google Research (2020)

### 🧠 Cómo se entrenó
T5 reformuló **todas las tareas de NLP como problemas de texto→texto**.  
Durante el entrenamiento, el modelo recibe una instrucción textual que describe la tarea, por ejemplo:
- “translate English to German: ...”
- “summarize: ...”
- “classify sentiment: ...”

Entrenado sobre el masivo **Colossal Clean Crawled Corpus (C4)**, que contiene cientos de gigabytes de texto web limpio.

### ⚙️ Enfoque arquitectónico
- Basado en una arquitectura **Encoder–Decoder** estándar de Transformer.
- Preentrenado con tareas multitarea unificadas bajo un mismo formato.

### 💪 Fortalezas
- Marco unificado para cualquier tarea de NLP.
- Permite *transfer learning* multitarea con un solo modelo.
- Versiones ligeras y grandes (T5-small a T5-11B).

### ⚠️ Debilidades
- Requiere mucha memoria y tiempo de entrenamiento.
- Puede ser menos preciso que modelos especializados en una sola tarea.
- No siempre entiende bien instrucciones ambiguas.

| Aspecto| Detalle|
| :----------------------------------------- | :----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Idioma original**| Inglés|
| **Corpus de entrenamiento**| C4 (Cleaned Common Crawl) — principalmente inglés|
| **Desempeño en español**| Aceptable, pero limitado al vocabulario inglés; no entiende bien acentos ni conjugaciones.|
| **Alternativas recomendadas para español** | 🟢 **mT5** (`google/mt5-small`, `mt5-base`, `mt5-large`) — entrenado en 101 idiomas (incluye español).<br>🟢 **ByT5** — variante byte-level, muy robusta con acentos y palabras fuera del vocabulario. |
| **Conclusión**| Usa **mT5** o **ByT5** para español o textos mixtos.|



In [28]:
t5 = pipeline(
    "text2text-generation",
    model="google-t5/t5-small",
    device=device
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [29]:
text = (
    "Machine learning enables computers to learn patterns from data. "
    "It is widely used in recommendation systems, fraud detection, "
    "and natural language processing."
)

prompt = f"summarize: {text}"

out = t5(prompt, max_new_tokens=40)

print("=== RESUMEN ===")
print(out[0]["generated_text"])

=== RESUMEN ===
machine learning enables computers to learn patterns from data . it is widely used in recommendation systems, fraud detection, and natural language processing .


In [30]:
context = (
    "Transformers were introduced in 2017 and revolutionized NLP "
    "through the use of self-attention mechanisms."
)

prompt = (
    "question: When were Transformers introduced? "
    f"context: {context}"
)

out = t5(prompt, max_new_tokens=20)

print("\n=== QA ===")
print(out[0]["generated_text"])


=== QA ===
2017


In [33]:
review = "This movie was fantastic and emotionally powerful."

prompt = f"sentiment: {review}"

out = t5(prompt, max_new_tokens=10)

print("\n=== SENTIMIENTO ===")
print(out[0]["generated_text"])


=== SENTIMIENTO ===
Das Gefühl: Dieser Film war fantastisch und


In [34]:
prompt = "translate English to German: I love natural language processing."

out = t5(prompt, max_new_tokens=20)

print(out[0]["generated_text"])

Ich liebe die natürliche Sprachenverarbeitung.


## ⚡ DistilBERT — Knowledge Distillation from BERT

**Empresa / patrocinio:** Hugging Face (2019)

### 🧠 Cómo se entrenó
DistilBERT se entrenó mediante **knowledge distillation**:  
un modelo “maestro” (BERT) entrena a un modelo más pequeño (“estudiante”) para imitar sus representaciones y predicciones.

El objetivo es conservar el 97% del rendimiento de BERT con 40% menos de parámetros.

### ⚙️ Enfoque arquitectónico
- Basado **solo en el encoder**, con menos capas (6 en lugar de 12).
- Conserva las atenciones y embeddings de BERT, ajustadas mediante distillation loss.

### 💪 Fortalezas
- Mucho más rápido y ligero (ideal para producción y dispositivos móviles).
- Rendimiento comparable en tareas de clasificación y NER.
- Open-source con soporte directo de Hugging Face.

### ⚠️ Debilidades
- Pérdida ligera de precisión frente a BERT completo.
- Menor capacidad de generalización en tareas complejas.
- No adecuado para tareas de generación.

| Aspecto| Detalle|
| :----------------------------------------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Idioma original**| Inglés|
| **Corpus de entrenamiento**| Knowledge distillation de `bert-base-uncased`|
| **Desempeño en español**| Pobre si se usa directamente (vocabulario inglés).|
| **Alternativas recomendadas para español** | 🟢 **DistilBETO** (si disponible, versiones reducidas de BETO).<br>🟢 **distilbert-base-multilingual-cased**: versión multilingüe reducida entrenada a partir de mBERT. |
| **Conclusión**| Usa **distilbert-base-multilingual-cased** para tareas rápidas en español.|


In [16]:
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [17]:
samples = [
    "I absolutely love how intuitive the new interface is!",
    "The latency is terrible and it keeps crashing."
]
sent_preds = sentiment(samples)

In [18]:
print("\n" + "="*70)
print("5) DistilBERT (Sentiment Analysis) — clasificación eficiente")
for s, p in zip(samples, sent_preds):
    print(f"- \"{s}\" → {p['label']} (score={p['score']:.3f})")


5) DistilBERT (Sentiment Analysis) — clasificación eficiente
- "I absolutely love how intuitive the new interface is!" → POSITIVE (score=1.000)
- "The latency is terrible and it keeps crashing." → NEGATIVE (score=0.987)


## 📊 Comparativa de modelos derivados del Transformer

| Modelo | Año | Empresa / Patrocinador | Enfoque arquitectónico | Idioma nativo | Variante recomendada para español | Fortalezas principales | Debilidades | Tareas ideales |
|:--|:--:|:--|:--|:--:|:--|:--|:--|:--|
| **BERT** | 2018 | Google AI | 🧱 Solo **Encoder** | 🇬🇧 Inglés | 🇪🇸 [`dccuchile/bert-base-spanish-wwm-cased`](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased) (BETO) / [`bert-base-multilingual-cased`](https://huggingface.co/bert-base-multilingual-cased) | Comprensión profunda de contexto bidireccional | No genera texto, alto costo de entrenamiento | Clasificación, NER, QA, embeddings |
| **GPT (1–5)** | 2018–2024 | OpenAI | 💬 Solo **Decoder** (autoregresivo) | 🇬🇧 Inglés | 🇪🇸 [`DeepESP/gpt2-spanish`](https://huggingface.co/DeepESP/gpt2-spanish) / modelos GPT-4/5 multilingües | Generación coherente, escritura creativa, diálogo | No bidireccional, posibles alucinaciones | Generación de texto, storytelling, chat |
| **BART** | 2019 | Meta AI | 🔄 **Encoder–Decoder** completo | 🇬🇧 Inglés | 🇪🇸 [`facebook/mbart-large-50`](https://huggingface.co/facebook/mbart-large-50) | Resumen, corrección y traducción | Modelo pesado, lento en inferencia | Summarization, denoising, traducción |
| **T5** | 2020 | Google Research | 🧮 **Encoder–Decoder** unificado (*text-to-text*) | 🇬🇧 Inglés | 🇪🇸 [`google/mt5-base`](https://huggingface.co/google/mt5-base) / [`google/byt5-base`](https://huggingface.co/google/byt5-base) | Multitarea texto→texto, adaptable | Costoso, sensible a instrucciones ambiguas | Traducción, resumen, clasificación, QA |
| **DistilBERT** | 2019 | Hugging Face | ⚡ Solo **Encoder** (destilado de BERT) | 🇬🇧 Inglés | 🇪🇸 [`distilbert-base-multilingual-cased`](https://huggingface.co/distilbert-base-multilingual-cased) | Ligero y rápido, ideal para producción | Menor precisión que BERT completo | Clasificación, análisis de sentimiento |


### 💡 Conclusión de la evolución

Tres líneas principales:
1) **Encoder-only (BERT):** comprensión profunda.  
2) **Decoder-only (GPT):** generación autoregresiva.  
3) **Encoder-Decoder (T5, BART):** traducción, resumen y multitarea.


## ⚡ Conclusiones

Los Transformers sustituyen la recurrencia por atención paralelizable, habilitando preentrenamiento masivo y alto rendimiento en múltiples tareas.  
La elección del subtipo depende de la tarea: comprensión (encoder-only), generación (decoder-only) o seq2seq (encoder-decoder).


## 🧪 Práctica 1: Traducción automática con un Transformer preentrenado

Usaremos `transformers` (HuggingFace) con un modelo encoder-decoder (p. ej., `Helsinki-NLP/opus-mt-es-en`).  
Pasos: cargar `pipeline("translation_xx_to_yy")`, traducir ejemplos y discutir resultados.


In [19]:
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")
textos = [
    "El modelo transformer revolucionó el procesamiento de lenguaje natural.",
    "Los mecanismos de atención permiten paralelismo total en entrenamiento."
]
trads = translator(textos)
for t, out in zip(textos, trads):
    print("\nES:", t)
    print("EN:", out["translation_text"])


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu



ES: El modelo transformer revolucionó el procesamiento de lenguaje natural.
EN: The transforming model revolutionized the processing of natural language.

ES: Los mecanismos de atención permiten paralelismo total en entrenamiento.
EN: The care mechanisms allow for total parallelism in training.


## 💬 Práctica 2: Generación autoregresiva con GPT-2 (decoder-only)

Demostraremos *prompting*, máscara causal y muestreo con temperatura.  
Nota: GPT-2 genera texto; su entrenamiento es **autoregresivo**: $P(w_1,\ldots,w_n)=\prod_t P(w_t \mid w_{<t})$.


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "La arquitectura Transformer cambió el campo del NLP porque"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=100,
        do_sample=True,
        temperature=0.9,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


La arquitectura Transformer cambió el campo del NLP porque esta poco, es que cuando nada no ocarina di bien su gente. Puede con l'entendido tardivementem parlano un mejorificaramentale e más donde anrocho a la jolie della hacia leyendo (Canto última).

Un trabajo


## 📚 Referencias

- Vaswani, A. et al. (2017). *Attention is All You Need.*  
- Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.*  
- Radford, A. et al. (2019). *Language Models are Unsupervised Multitask Learners.*  
- Lewis, M. et al. (2019). *BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation.*  
- Raffel, C. et al. (2020). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (T5).*  
- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT.*
